# Gradient Descent via API (batch calls)

This notebook demonstrates running gradient descent on the client while calling the API to obtain the loss and analytical gradient at each step using `/evaluate-with-visualization`.

It updates a side-by-side visualization every few iterations:
- Left: feature space (dataset points + decision boundary from current params)
- Right: loss landscape slice over (w1, w2) with the moving optimizer point


In [ ]:
import time
import requests
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

from app.functions import quadratic_binary_loss, quadratic_predict_proba

URL = "http://localhost:8000/evaluate-with-visualization"

def plot_pair(params, points, y_true, loss_fn, predict_proba_fn, w_idx=(1,2), grid_radius=2.0):
    p = np.asarray(params)
    fig, (axl, axr) = plt.subplots(1,2, figsize=(10,5))
    # left: points + decision boundary
    axl.scatter(points[:,0], points[:,1], c=y_true, cmap='coolwarm', s=25, edgecolor='k')
    xs = np.linspace(points[:,0].min()-0.5, points[:,0].max()+0.5, 200)
    ys = np.linspace(points[:,1].min()-0.5, points[:,1].max()+0.5, 200)
    XX, YY = np.meshgrid(xs, ys)
    grid = np.column_stack([XX.ravel(), YY.ravel()])
    probs = predict_proba_fn(p, grid)
    ZZ = probs.reshape(XX.shape)
    axl.contour(XX, YY, ZZ, levels=[0.5], colors=['k'])
    axl.set_title('Feature space (points + decision boundary)')
    # right: loss landscape slice
    i, j = w_idx
    center_i, center_j = p[i], p[j]
    grid_vals = np.linspace(center_i - grid_radius, center_i + grid_radius, 120)
    Gx, Gy = np.meshgrid(grid_vals, grid_vals)
    losses = np.empty_like(Gx)
    for a in range(Gx.shape[0]):
        for b in range(Gx.shape[1]):
            q = p.copy()
            q[i] = Gx[a,b]
            q[j] = Gy[a,b]
            losses[a,b] = loss_fn(q)
    im = axr.imshow(losses, extent=(grid_vals.min(), grid_vals.max(), grid_vals.min(), grid_vals.max()), origin='lower', cmap='viridis')
    axr.scatter([p[i]], [p[j]], color='red', s=50)
    axr.set_xlabel(f'w[{i}]'); axr.set_ylabel(f'w[{j}]')
    axr.set_title('Loss landscape (slice)')
    fig.colorbar(im, ax=axr)
    plt.tight_layout()
    return fig

# GD loop that calls the API for gradient every iteration
def gd_via_api(initial_params, lr=0.5, steps=80, plot_every=5):
    params = np.asarray(initial_params, dtype=float).copy()
    history = [params.copy()]
    last_points = None
    last_y = None
    for t in range(steps):
        payload = {"problem_id": "quadratic_binary", "x": params.tolist()}
        try:
            r = requests.post(URL, json=payload, timeout=5.0)
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            print('API request failed at step', t, e)
            break
        grad = np.asarray(data['gradient'], dtype=float)
        loss = float(data['y'])
        points = np.asarray(data['points']) if data.get('points') is not None else None
        y_true = None
        if points is not None:
            # load true labels from local dataset for plotting; lighter than returning labels
            from app.datasets import get_dataset
            Xd, yd = get_dataset('quadratic_binary')
            y_true = yd
            last_points = points
            last_y = y_true
        # gradient descent update
        params = params - lr * grad
        history.append(params.copy())
        # plotting
        if (t % plot_every) == 0:
            fig = plot_pair(params, last_points if last_points is not None else Xd, last_y if last_y is not None else yd, quadratic_binary_loss, quadratic_predict_proba, w_idx=(1,2))
            clear_output(wait=True)
            display(fig)
            plt.close(fig)
            print(f'step={t}, loss={loss:.4f}')
            time.sleep(0.05)
    return history

# Run the GD via API example
init = np.zeros(6)
init[3] = -1.0  # small quadratic bias to start
hist = gd_via_api(init, lr=0.6, steps=120, plot_every=3)
